In [ ]:
%cd ..

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# Generate random text data
from spesia_ner.datasets import ClinicalRecordsDataset
from spesia_ner.data_models import AgentGeneratedRecord

load_dotenv()

prompt = [
    {
        'role': 'system',
        'content': 'Generate a fake news paragraph talking about unimportant and uninteresting topics. Generate no more than 3 sentences. Include fake people names, fake locations and fake organizations. Annotate the generated paragraph with tags according to the requested format in JSON. The entities include only: PERSON, ORGANIZATION, LOCATION.'
    }
]

prompt = {'messages': prompt}

model_name = 'gpt-5.1'
reasoning_effort = None
batch_size = 10
sample_size = 100
records = []

model = init_chat_model(
    model_name,
    reasoning_effort=reasoning_effort,
)

agent = create_agent(
    model=model,
    response_format=AgentGeneratedRecord, 
)

while len(records) < sample_size:
    results = agent.batch([prompt for _ in range(batch_size)])
    results = [r['structured_response'].to_record() for r in results]
    records.extend(results)


fake_dataset = ClinicalRecordsDataset()
fake_dataset.records = records

fake_dataset.export(path='data/dummy_data/fake_data.jsonl', format='jsonl', include_annotations=True)